# 08 — Context Managers and Resource Safety

Context managers provide structured setup and cleanup. They are essential for file handles, locks, temporary resources, and database transactions. Understand both with blocks and contextlib.contextmanager.

*Intermediate · 75–120 min*

## You will learn

- Explain the __enter__/__exit__ protocol
- Use with for safe resource cleanup
- Build a context manager with contextlib.contextmanager
- Handle failure without hiding exceptions

## Examples

### 1. The with statement

A context manager is entered before the indented block and exited afterward, even if an exception occurs. File objects close automatically using with. This avoids leaking file descriptors and makes cleanup visible.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    path = Path(directory) / "notes.txt"
    with path.open("w", encoding="utf-8") as stream:
        stream.write("hello\nworld\n")
    print(path.read_text(encoding="utf-8"))

**Reasoning prompt:** What happens if writing raises an exception inside the with block?

### 2. A class-based context manager

__enter__ returns the object used after as; __exit__ receives exception information and runs on normal and abnormal exits. Returning False (or None) allows exceptions to propagate. Returning True suppresses them, which should be deliberate.

In [ ]:
class LabelledBlock:
    def __init__(self, label):
        self.label = label
    def __enter__(self):
        print("begin", self.label)
        return self
    def __exit__(self, exc_type, exc, traceback):
        print("end", self.label)
        return False

with LabelledBlock("report"):
    print("inside")

**Reasoning prompt:** Why is suppressing all exceptions from __exit__ generally unsafe?

### 3. The contextmanager decorator

contextlib.contextmanager turns a generator with one yield into a context manager. Code before yield sets up a resource; code in finally cleans it up. The try/finally is important if body code raises.

In [ ]:
from contextlib import contextmanager

@contextmanager
def status(label):
    print("opening:", label)
    try:
        yield {"name": label}
    finally:
        print("closing:", label)

with status("demo") as info:
    print(info["name"])

**Reasoning prompt:** Would the closing message still print if the body raises?

### 4. Temporary resources in tests

TemporaryDirectory gives you a disposable directory that is deleted upon leaving its context. Tests can safely create files without overwriting course data or leaving stray artifacts in the repository.

In [ ]:
from tempfile import TemporaryDirectory
from pathlib import Path
with TemporaryDirectory() as folder:
    temporary_file = Path(folder) / "result.csv"
    temporary_file.write_text("name,score\nAda,90\n", encoding="utf-8")
    print(temporary_file.exists(), temporary_file.read_text(encoding="utf-8").splitlines())

**Reasoning prompt:** Why is a temporary directory more reliable than using a hardcoded path in a notebook?

## Exercises

**Exercise 1:** Build a timer context manager using time.perf_counter.

In [ ]:
# Exercise 1: implement your answer here.
# Add one normal-case test and one edge-case test.

**Exercise 2:** Create a logging context manager that reports entry and exit.

In [ ]:
# Exercise 2: implement your answer here.
# Add one normal-case test and one edge-case test.

**Exercise 3:** Write and read a temporary UTF-8 file with nested with blocks.

In [ ]:
# Exercise 3: implement your answer here.
# Add one normal-case test and one edge-case test.

**Exercise 4:** Implement __exit__ so that exceptions propagate while cleanup still happens.

In [ ]:
# Exercise 4: implement your answer here.
# Add one normal-case test and one edge-case test.

## Common mistakes

- A context manager must clean up on both success and failure.
- Returning True from __exit__ suppresses exceptions.
- Avoid hardcoded output paths that could overwrite real files.

**Next:** [09 — Functional Tools, Sorting, and itertools](09_functional_tools_and_itertools.ipynb)